# 01 — Marketing Campaign Data Inspection

This notebook loads the supplied ZIP archive, inventories its contents, reads the five Parquet tables, and performs first-pass structural and data-quality checks. It does not assume row counts or business results in advance.

In [ ]:
%pip -q install pyarrow pandas

import zipfile
from pathlib import Path
import pandas as pd
from IPython.display import display

print('Dependencies ready')

## 1. Upload the dataset ZIP

Upload the same ZIP archive supplied for this project. Keep the original archive unchanged.

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Please upload the project ZIP file.')

archive_path = Path('/content') / zip_name
print(f'Uploaded: {zip_name} | {archive_path.stat().st_size:,} bytes')

In [ ]:
extract_dir = Path('/content/marketing_campaign_data')
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(archive_path) as zf:
    bad_member = zf.testzip()
    if bad_member:
        raise RuntimeError(f'ZIP integrity check failed at: {bad_member}')
    zf.extractall(extract_dir)
    inventory = pd.DataFrame([
        {'file': info.filename, 'size_bytes': info.file_size}
        for info in zf.infolist() if not info.is_dir()
    ])

display(inventory)
print('ZIP integrity check passed')

## 2. Load Parquet tables

The SQL schema declares five tables. This cell reads the actual files and reports observed row counts and columns.

In [ ]:
parquet_paths = {p.stem: p for p in extract_dir.rglob('*.parquet')}
expected_tables = {'channels', 'campaigns', 'ad_groups', 'daily_performance', 'conversions'}
missing_tables = expected_tables - set(parquet_paths)
if missing_tables:
    raise FileNotFoundError(f'Missing expected Parquet tables: {sorted(missing_tables)}')

tables = {name: pd.read_parquet(path) for name, path in parquet_paths.items()}
table_summary = pd.DataFrame([
    {'table': name, 'rows': len(df), 'columns': len(df.columns), 'memory_mb': round(df.memory_usage(deep=True).sum() / 1024**2, 3)}
    for name, df in tables.items()
]).sort_values('table').reset_index(drop=True)

display(table_summary)
for name, df in tables.items():
    print(f'\\n--- {name} | shape={df.shape} ---')
    print(df.dtypes.to_string())
    display(df.head(3))

## 3. First-pass data-quality checks

These checks are diagnostic only. Do not remove records until the reason and business impact are understood.

In [ ]:
quality_rows = []
for name, df in tables.items():
    quality_rows.append({
        'table': name,
        'rows': len(df),
        'duplicate_rows': int(df.duplicated().sum()),
        'total_missing_cells': int(df.isna().sum().sum()),
        'columns_with_missing': int(df.isna().any().sum())
    })
quality_summary = pd.DataFrame(quality_rows).sort_values('table').reset_index(drop=True)
display(quality_summary)

In [ ]:
# Key uniqueness and relationship checks based on the supplied SQL schema
primary_keys = {
    'channels': 'channel_id',
    'campaigns': 'campaign_id',
    'ad_groups': 'ad_group_id',
    'daily_performance': 'performance_id',
    'conversions': 'conversion_id'
}

key_checks = []
for table_name, key in primary_keys.items():
    series = tables[table_name][key]
    key_checks.append({
        'table': table_name,
        'primary_key': key,
        'null_keys': int(series.isna().sum()),
        'duplicate_key_values': int(series.duplicated().sum())
    })
display(pd.DataFrame(key_checks))

foreign_keys = [
    ('campaigns', 'channel_id', 'channels', 'channel_id'),
    ('ad_groups', 'campaign_id', 'campaigns', 'campaign_id'),
    ('daily_performance', 'ad_group_id', 'ad_groups', 'ad_group_id'),
    ('conversions', 'ad_group_id', 'ad_groups', 'ad_group_id'),
    ('conversions', 'first_touch_channel_id', 'channels', 'channel_id'),
    ('conversions', 'last_touch_channel_id', 'channels', 'channel_id')
]

fk_checks = []
for child, child_col, parent, parent_col in foreign_keys:
    child_values = tables[child][child_col].dropna()
    parent_values = set(tables[parent][parent_col].dropna().unique())
    unmatched = int((~child_values.isin(parent_values)).sum())
    fk_checks.append({'relationship': f'{child}.{child_col} -> {parent}.{parent_col}', 'unmatched_child_rows': unmatched})
display(pd.DataFrame(fk_checks))

## 4. Save inspection outputs

Export small summary files for reproducibility. Do not export or commit the raw tables from this notebook.

In [ ]:
output_dir = Path('/content/inspection_outputs')
output_dir.mkdir(exist_ok=True)
table_summary.to_csv(output_dir / 'table_summary.csv', index=False)
quality_summary.to_csv(output_dir / 'data_quality_summary.csv', index=False)
pd.DataFrame(key_checks).to_csv(output_dir / 'primary_key_checks.csv', index=False)
pd.DataFrame(fk_checks).to_csv(output_dir / 'foreign_key_checks.csv', index=False)

print(f'Inspection summaries saved to: {output_dir}')

## 5. Record findings before moving on

After running every cell, record the observed table sizes, key-integrity results, missing values, duplicate counts, date ranges, and any anomalies. Do not calculate or publish CTR, CPC, CPA, ROAS, or ROI until the underlying fields and metric definitions have been reconciled.